# 05 — Similarity facts

Turn each DayHike in one region into a row of similarity facts, one Facet at a
time, and write that table next to the Catalog. `05a_similarity.ipynb` compares
DayHikes from it. This is the notebook design phase of ADR-0011 (proposed):
production Ranking, the CLI and the Catalog do not change.

**Region:** DayHikes whose Trailhead lies in lat 42.3–46.9, lon 13.3–19.5
(Slovenia + Croatia: Julian Alps, karst, Velebit, the coast, the Pannonian
lowland). `load_region` reads the GeometryStore the way `04a_geometry.ipynb`
does (vertices, amenities, dayhikes, ways as GeoDataFrames) with the Catalog's
`feature_vectors`, Trailhead and Difficulty merged onto `dayhikes`. Rows are
filtered by region DayHike id in SQL; the full GeometryStore is ~19 GB.

Open `data/catalog.sqlite` plus `data/catalog.geometry.sqlite` when both exist.
Otherwise (and always under pytest) build the committed OSM fixture
`tests/fixtures/catalog.osm`, whose DayHikes all lie inside the same box.

**Facets here:** Effort, Altitude.

A missing fact stays missing (NaN), never 0: a DayHike without elevation is
not a flat DayHike.

In [ ]:
import os
import tempfile
from pathlib import Path

import geopandas as gpd
import pandas as pd

from dayhike import build_catalog, geometry_store_path_for
from dayhike.similarity import (
    altitude_facts,
    effort_facts,
    load_region,
    similarity_facts_path_for,
)

if "PYTEST_CURRENT_TEST" in os.environ:
    import matplotlib as mpl

    mpl.use("Agg")
import matplotlib.pyplot as plt


def show_fig(fig):
    try:
        from IPython.display import display

        display(fig)
    except ImportError:
        pass
    plt.close(fig)


repo = Path.cwd().resolve()
for candidate in (repo, *repo.parents):
    fixture = candidate / "tests" / "fixtures" / "catalog.osm"
    if fixture.is_file():
        repo = candidate
        break
else:
    raise FileNotFoundError("tests/fixtures/catalog.osm")

local = repo / "data" / "catalog.sqlite"
if (
    local.is_file()
    and geometry_store_path_for(local).is_file()
    and "PYTEST_CURRENT_TEST" not in os.environ
):
    catalog_path = local
else:
    catalog_path = Path(tempfile.mkdtemp()) / "catalog.sqlite"
    build_catalog(fixture, catalog_path)

REGION_LAT = (42.3, 46.9)
REGION_LON = (13.3, 19.5)
vertices, amenities, dayhikes, ways = load_region(
    catalog_path, lat_range=REGION_LAT, lon_range=REGION_LON
)
print("catalog", catalog_path)
for name, frame in (
    ("vertices", vertices),
    ("amenities", amenities),
    ("dayhikes", dayhikes),
    ("ways", ways),
):
    print(f"region {name}\t{len(frame)}")
print("name_origin", dayhikes["name_origin"].value_counts().to_dict())

## Effort

How much work the day is. Three facts per DayHike:

- `length_m`: the Catalog length of the walked vertex chain.
- `elevation_gain_m`: metres climbed along the chain. OSM way order is
  arbitrary, so a point-to-point DayHike stored "downhill" would show almost no
  climb. The fact is the larger of the climb and the descent along the chain,
  which is the same in both directions. On a loop the two are equal. Vertices
  without `ele` are skipped (the Catalog does the same); no `ele` at all leaves
  the gain missing.
- `difficulty_rank`: Difficulty as the SAC rank 1–6 (`hiking` 1 …
  `difficult_alpine_hiking` 6), from the strictest `sac_scale` on member ways.
  Missing when no member way is tagged, which is common.

In [ ]:
effort = effort_facts(dayhikes, vertices)
print(effort.describe().round(1).to_string())
print("effort missing share")
print(effort.isna().mean().round(3).to_string())

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
axes[0].hist(effort["length_m"].dropna() / 1000, bins=40, color="#0f766e")
axes[0].set_xlabel("length (km)")
axes[1].hist(effort["elevation_gain_m"].dropna(), bins=40, color="#c2410c")
axes[1].set_xlabel("elevation gain (m)")
ranks = effort["difficulty_rank"].value_counts().sort_index()
axes[2].bar(ranks.index.astype(int).astype(str), ranks.to_numpy(), color="#7c3aed")
axes[2].set_xlabel("Difficulty rank (SAC)")
for ax in axes:
    ax.set_ylabel("DayHikes")
fig.suptitle("Effort facts in the region")
fig.tight_layout()
show_fig(fig)

## Altitude

How high the DayHike goes, apart from how much it climbs. Two facts, read from
the Catalog's `feature_vectors` (FABDEM when `sample-dem` has run, else OSM
`ele`):

- `highest_m`: the highest vertex.
- `lowest_m`: the lowest vertex. A coastal climb from sea level and a walk from
  a 1500 m pass can reach the same top.

Both are the same whichever way the DayHike is walked.

In [ ]:
altitude = altitude_facts(dayhikes)
print(altitude.describe().round(1).to_string())
print("altitude missing share")
print(altitude.isna().mean().round(3).to_string())

fig, ax = plt.subplots(figsize=(7, 3.5))
bins = 40
ax.hist(altitude["lowest_m"].dropna(), bins=bins, alpha=0.6, color="#1d4ed8", label="lowest")
ax.hist(altitude["highest_m"].dropna(), bins=bins, alpha=0.6, color="#c2410c", label="highest")
ax.set_xlabel("elevation (m)")
ax.set_ylabel("DayHikes")
ax.set_title("Altitude facts in the region")
ax.legend()
fig.tight_layout()
show_fig(fig)

## Similarity facts table

One row per region DayHike, indexed by DayHike id: `name`, `name_origin`, the
Trailhead and the DayHike line (for maps in `05a`), then every Facet's facts.
It is written as a pandas pickle next to the Catalog,
`{catalog}.similarity_facts.pkl` (`data/catalog.similarity_facts.pkl` for the
local Catalog). It is regenerable and gitignored (`*.similarity_facts.pkl`).

In [ ]:
facts = gpd.GeoDataFrame(
    dayhikes.set_index("id")[
        ["name", "name_origin", "trailhead_latitude", "trailhead_longitude", "geometry"]
    ]
    .join(effort)
    .join(altitude),
    geometry="geometry",
    crs=dayhikes.crs,
)
facts_path = similarity_facts_path_for(catalog_path)
facts.to_pickle(facts_path)
print("similarity_facts", facts_path)
print(f"similarity_facts_rows\t{len(facts)}")
print("columns", list(facts.columns))